# 🧭 Nextify Course Notebook – Agent 1 (Idea Intake + Brainstorming)
_Continuation from the earlier "Nextify – Your Innovation Pal" notebook in the Google course reading branch. Kaggle-ready and GitHub-friendly._

This notebook mirrors the course practice flow while using the **Google ADK** (Gemini-first) plus a **Switch Agent** that can fall back to OpenAI. We will:
- Capture the Idea Form (user-filled) for the idea track.
- Stand up the Switch Agent as an external service callable by any agent.
- Run the brainstorming sub-agents (market, crazy ideas, synthesis) with a revision loop to produce a Product Snapshot hand-off.


## 0. Environment setup (Kaggle + GitHub, ADK-aligned)
- Uses Google ADK (`google-genai`) for Gemini calls; mirrors the course reading notebooks.
- Switch Agent prompts the subscriber before routing to OpenAI when Gemini hits a token limit or similar error.
- Carries a running continuity summary so context is preserved across providers.
- If running on GitHub/Colab, enable the install cell below; Kaggle images already include ADK.


In [ ]:
# Toggle installation for GitHub/Colab (Kaggle already has google-genai installed)
INSTALL_DEPS = False  # set to True when running outside Kaggle
if INSTALL_DEPS:
    %pip install -q google-genai google-generativeai google-ai-generativelanguage google-auth google-auth-httplib2 google-auth-oauthlib openai python-dotenv


### API keys (Gemini + OpenAI)
- Ensure `GOOGLE_API_KEY` is set (Kaggle: add to notebook secrets; GitHub: use `.env`).
- Optional: set `OPENAI_API_KEY` to allow fallback when Gemini hits limits.
- This cell mirrors the setup pattern used in the course reading notebooks.


In [ ]:
import os, json, textwrap
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Callable

# Load environment variables for local/github runs
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not GOOGLE_API_KEY:
    print("⚠️ GOOGLE_API_KEY is not set. Add it via Kaggle secrets or a local .env file.")
if not OPENAI_API_KEY:
    print("ℹ️ OPENAI_API_KEY not set; OpenAI fallback will be disabled unless provided.")


In [ ]:
# ADK / LLM clients
try:
    from google import genai  # google-genai (ADK)
    import google.generativeai as genai_legacy  # legacy client still used by some ADK helpers
    genai.configure(api_key=GOOGLE_API_KEY)
    ADK_AVAILABLE = True
except Exception:
    genai = None
    genai_legacy = None
    ADK_AVAILABLE = False

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

GEMINI_MODEL = "gemini-1.5-pro-latest"
OPENAI_MODEL = "gpt-4o-mini"


## 1. Idea Form (copy from Notebook 0)
- Paste the values you captured in the "Nextify – Your Innovation Pal" notebook.
- Keep the structure identical to the course reading notebooks so downstream agents stay schema-aligned.


In [ ]:
idea_form = {
    # 🔁 Replace the sample values with your own (from Notebook 0).
    "idea_name": "Nextify - Product Strategy Copilot",
    "persona": "Founder building AI-first PM toolkit",
    "problem": "Quickly turn raw ideas into prioritized product snapshots with minimal manual research.",
    "target_users": "Early-stage founders and PMs",
    "solution_approach": "Multi-agent system for intake, brainstorming, market sizing, and snapshot synthesis.",
    "industry": "B2B SaaS / Product Management",
    "technology_trends": "Agentic workflows, LLM routing, AI copilots for PM",
    "constraints": "Kaggle runtime; Gemini-first with OpenAI fallback",
    "success_metric": "Actionable Product Snapshot with top 1–2 concepts and TAM/SAM/SOM notes",
}

print(json.dumps(idea_form, indent=2))


## 2. Switch Agent (Gemini → OpenAI) – shared external service
- Lives outside the per-agent architecture; every agent calls it when invoking LLMs.
- Tries Gemini (ADK) first; on token limit/error, asks the subscriber whether to switch.
- When switching, it forwards a continuity summary plus recent outputs for that agent.
- Tracks the provider chain used so far for observability.


In [ ]:
class LLMError(Exception):
    ...

@dataclass
class LLMConfig:
    primary: str = "gemini"  # 'gemini' or 'openai'
    gemini_model: str = GEMINI_MODEL
    openai_model: str = OPENAI_MODEL

@dataclass
class SwitchState:
    continuity_summary: str = ""
    outputs: List[str] = None
    providers: List[str] = None

    def __post_init__(self):
        self.outputs = self.outputs or []
        self.providers = self.providers or []

class LLMSwitchAgent:
    """Shared router that wraps Gemini (ADK) and OpenAI and preserves context."""

    def __init__(self, cfg: LLMConfig, subscriber_allows_switch: bool = True, confirm_fn: Optional[Callable[[str], bool]] = None):
        self.cfg = cfg
        self.subscriber_allows_switch = subscriber_allows_switch
        self.confirm_fn = confirm_fn
        self._gemini_client = genai.Client(api_key=GOOGLE_API_KEY) if genai else None
        self._openai_client = OpenAI(api_key=OPENAI_API_KEY) if OpenAI else None

    def _call_gemini_via_adk(self, prompt: str) -> str:
        if not self._gemini_client:
            raise LLMError("Gemini client not available")
        # ADK-style: create a chat session and send the prompt
        session = self._gemini_client.chats.create(
            model=self.cfg.gemini_model,
            system_instruction="You are part of the Nextify multi-agent ADK pipeline.",
        )
        resp = session.send_message(prompt)
        return getattr(resp, "text", str(resp))

    def _call_openai(self, prompt: str) -> str:
        if not self._openai_client:
            raise LLMError("OpenAI client not available")
        resp = self._openai_client.chat.completions.create(
            model=self.cfg.openai_model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
        )
        return resp.choices[0].message.content

    def _subscriber_confirms(self, reason: str) -> bool:
        if self.confirm_fn:
            return bool(self.confirm_fn(reason))
        return bool(self.subscriber_allows_switch)

    def run(self, prompt: str, state: SwitchState):
        """Try Gemini first; if a limit/error occurs and subscriber consents, switch to OpenAI."""
        try:
            text = self._call_gemini_via_adk(prompt)
            state.providers.append("gemini")
            return text, state
        except Exception as e:
            state.providers.append("gemini:error")
            if not self._subscriber_confirms(str(e)):
                raise LLMError(f"Gemini failed and switch not allowed: {e}")
            try:
                prompt_with_summary = textwrap.dedent(f"""
                Previous summary: {state.continuity_summary or 'N/A'}
                Prior outputs: {json.dumps(state.outputs[-3:], ensure_ascii=False)}
                Continue: {prompt}
                """)
                text = self._call_openai(prompt_with_summary)
                state.providers.append("openai")
                return text, state
            except Exception as e2:
                state.providers.append("openai:error")
                raise LLMError(f"Both providers failed: {e} | {e2}")

    def update_summary(self, state: SwitchState, new_output: str) -> SwitchState:
        state.outputs.append(new_output)
        summary_slice = state.outputs[-3:]
        state.continuity_summary = " | ".join([o[:200] for o in summary_slice])
        return state


## 3. Brainstorming sub-agents (ADK style)
- **MarketAnalysisSubAgent**: TAM/SAM/SOM + market signals for the exact idea.
- **CrazyIdeaSubAgent**: playful cross-industry mashups that consider trends.
- **SynthesisSubAgent**: merges both, scores candidates, and proposes evaluated options.
Each sub-agent runs through the Switch Agent so they inherit continuity and can recover on provider limits. Prompts are written in the ADK style from the reading notebooks.


In [ ]:
@dataclass
class AgentSpec:
    name: str
    instruction: str
    model: str = GEMINI_MODEL
    tools: Optional[List[str]] = None  # placeholders for ADK tool wiring

market_agent = AgentSpec(
    name="MarketAnalysisSubAgent",
    instruction=textwrap.dedent("""
    You are MarketAnalysisSubAgent. Build a concise TAM/SAM/SOM breakdown for the idea.
    Include: target users, assumptions, pricing anchor, TAM/SAM/SOM numbers (rough), top risks, and demand signals.
    Return markdown with bullet lists and a short table.
    """),
)

crazy_agent = AgentSpec(
    name="CrazyIdeaSubAgent",
    instruction=textwrap.dedent("""
    You are CrazyIdeaSubAgent. Make bold cross-industry combinations.
    Blend at least two adjacent or surprising industries, grounded with current tech/industry trends.
    Return 3 concepts with hooks, differentiation, and feasibility notes.
    """),
)

synthesis_agent = AgentSpec(
    name="SynthesisSubAgent",
    instruction=textwrap.dedent("""
    You are SynthesisSubAgent. Merge market analysis + wild ideas to propose 3-5 evaluated concepts.
    For each concept provide: one-liner, user value, feasibility, risks, and an overall score (1-5).
    End with a Product Snapshot draft that extends the Idea Form with: Problem, Users, Solution, Differentiation,
    TAM/SAM/SOM, Risks, Provider chain used, analytics table of evaluated ideas, pros/cons, and the user-selected concept.
    """),
)

def render_prompt(agent: AgentSpec, idea: Dict[str, Any], market: str = "", wild: str = "", user_feedback: str = "") -> str:
    base = [agent.instruction, f"Idea: {json.dumps(idea, ensure_ascii=False)}"]
    if market:
        base.append(f"Market analysis input: {market}")
    if wild:
        base.append(f"Crazy ideas input: {wild}")
    if user_feedback:
        base.append(f"Subscriber feedback: {user_feedback}")
    base.append("Return concise markdown and clearly label the Product Snapshot section.")
    return "\n".join(base)


## 4. Brainstorming orchestrator (with revision loop)
- Runs the three sub-agents through the Switch Agent (Gemini-first via ADK).
- Maintains continuity summary and provider chain for transparency.
- Allows up to two revisions with user feedback; final output is a Product Snapshot to hand off downstream.
- Mirrors the step-by-step loops in the course reading notebooks so you can confirm each agent.


In [ ]:
def run_brainstorming(
    idea: Dict[str, Any],
    switch_agent: LLMSwitchAgent,
    state: SwitchState,
    user_feedback: str = "",
    revision_count: int = 0,
    max_revisions: int = 2,
):
    if revision_count > max_revisions:
        raise ValueError("Revision limit reached")

    market_prompt = render_prompt(market_agent, idea)
    market_out, state = switch_agent.run(market_prompt, state)
    state = switch_agent.update_summary(state, market_out)

    crazy_prompt = render_prompt(crazy_agent, idea)
    crazy_out, state = switch_agent.run(crazy_prompt, state)
    state = switch_agent.update_summary(state, crazy_out)

    synth_prompt = render_prompt(synthesis_agent, idea, market=market_out, wild=crazy_out, user_feedback=user_feedback)
    synth_out, state = switch_agent.run(synth_prompt, state)
    state = switch_agent.update_summary(state, synth_out)

    product_snapshot = synth_out
    revision_count = min(revision_count, max_revisions)
    return product_snapshot, state.providers, state.continuity_summary, revision_count, state


## 5. First pass (no feedback yet)
Run this cell to generate the initial Product Snapshot. If Gemini hits a limit, the Switch Agent will prompt the subscriber to approve an OpenAI retry and will pass along the running summary.


In [ ]:
subscriber_allows_switch = True  # set False to disable OpenAI fallback
cfg = LLMConfig(primary="gemini")
switch_agent = LLMSwitchAgent(cfg, subscriber_allows_switch=subscriber_allows_switch)
state = SwitchState()

product_snapshot, providers_used, continuity_summary, revision_count, state = run_brainstorming(
    idea_form,
    switch_agent,
    state,
    user_feedback="",
    revision_count=0,
)

print("Providers used:", providers_used)
print("Continuity summary (truncated):", continuity_summary[:300])
print("Product Snapshot (draft):", product_snapshot)


## 6. User confirmation loop (max 2 revisions)
1. Set `user_feedback` below (choose a concept, add constraints, or request a tweak).
2. Re-run the cell to iterate. The Switch Agent retains summaries and provider chain across revisions.


In [ ]:
user_feedback = ""  # e.g., "Pick concept C2, emphasize onboarding risk, shorten TAM notes"
revision_count += 1
product_snapshot, providers_used, continuity_summary, revision_count, state = run_brainstorming(
    idea_form,
    switch_agent,
    state,
    user_feedback=user_feedback,
    revision_count=revision_count,
)

print("Revision:", revision_count)
print("Providers used:", providers_used)
print("Continuity summary (truncated):", continuity_summary[:300])
print("Product Snapshot (updated):", product_snapshot)
